# e-SNLI Frequency Filter — Lowest-Frequency Feature Selection

In [1]:
import sys, os, re

sys.path.insert(0, os.path.dirname(os.getcwd()))

import random

import torch
from src.configs import SAEConfig
from src.denoiser import Denoiser
from src.feature import create_features
from src.neuronpedia_client import NeuronpediaClient, build_sae_id
from src.utils.visualization import ActivationHeatmap

In [2]:
DATA_PATH = "../data/activations/esnli-l40-65k.pt"

data = torch.load(DATA_PATH, weights_only=False)

sae_encodings = data["sae_encodings"]
recon_stats = data["recon_stats"]
sequences = data["sequence"]
prompt_char_lens = data["prompt_lens"]
generation_token_ids = data.get("generation_token_ids")
dataset_info = data["dataset_info"]
sae_cfg_dict = data["sae_config"]

sae_config = SAEConfig(**sae_cfg_dict)

n_prompts = len(sae_encodings)
total_tokens = sum(enc.shape[0] for enc in sae_encodings)
n_features = sae_encodings[0].shape[1]
mean_fvu = sum(s["fvu"] for s in recon_stats) / len(recon_stats)
mean_l0 = sum(s["l0"] for s in recon_stats) / len(recon_stats)

print(f"Prompts:          {n_prompts}")
print(f"Total tokens:     {total_tokens}")
print(f"Features (d_sae): {n_features}")
print(f"Mean FVU:         {mean_fvu:.4f}")
print(f"Mean L0:          {mean_l0:.1f}")
print(f"SAE config:       {sae_config}")

Prompts:          500
Total tokens:     72852
Features (d_sae): 65536
Mean FVU:         0.0229
Mean L0:          57.6
SAE config:       SAEConfig(repo_id='google/gemma-scope-2-27b-it', sae_type='resid_post', layer=40, width='65k', l0='medium')


In [3]:
MODEL_NAME = "google/gemma-3-27b-it"

import sys
from huggingface_hub import login
from transformers import AutoTokenizer

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")
    login(token=hf_token)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=True)

masks = []
for gen_tok_ids in generation_token_ids:
    toks = tokenizer.convert_ids_to_tokens(gen_tok_ids)
    masks.append(Denoiser.make_special_token_mask(toks))

print(f"Built {len(masks)} special-token masks")
print(f"Example mask shape: {masks[0].shape}")
print(f"Example mask (first 10): {masks[0][:10]}")
print(f"Special tokens in first sample: {masks[0].sum().item()}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Built 500 special-token masks
Example mask shape: torch.Size([149])
Example mask (first 10): tensor([False, False, False, False, False, False, False, False, False, False])
Special tokens in first sample: 1


In [4]:
freq = Denoiser.compute_frequency_filter(
    sae_encodings,
    threshold=0.0,
    special_token_masks=masks,
)

print(f"Frequency tensor shape: {freq.shape}")

Computing frequency filter: 100%|██████████| 500/500 [00:26<00:00, 18.76it/s]

Frequency tensor shape: torch.Size([65536])


In [5]:
seed = random.randint(0, 2**32 - 1)
random.seed(seed)
sample_idx = random.randint(0, n_prompts - 1)
print(f"Random seed:          {seed}")
print(f"Sampled prompt index: {sample_idx}")

sample_sparse = sae_encodings[sample_idx]
sample_acts = sample_sparse.to_dense().float()

gen_tok_ids = generation_token_ids[sample_idx]
tokens = tokenizer.convert_ids_to_tokens(gen_tok_ids)

if len(tokens) != sample_acts.shape[0]:
    print(
        f"Token count mismatch: tokenizer produced {len(tokens)}, "
        f"activations have {sample_acts.shape[0]}. "
        f"Falling back to positional labels."
    )
    tokens = [f"t_{i}" for i in range(sample_acts.shape[0])]

gen_text = sequences[sample_idx][prompt_char_lens[sample_idx]:]
ground_truth = dataset_info["ground_truths"][sample_idx]

print(f"Ground truth label: {ground_truth}")
print(f"Generation tokens:  {sample_acts.shape[0]}")
print(f"Token count:        {len(tokens)}")
print(f"First 10 tokens:    {tokens[:10]}")
print(f"Generation text:    {gen_text[:200]}...")

Random seed:          2478648038
Sampled prompt index: 312
Ground truth label: contradiction
Generation tokens:  145
Token count:        145
First 10 tokens:    ['\n', '<', 'reason', 'ing', '>', 'The', '▁premise', '▁describes', '▁a', '▁scene']
Generation text:    
<reasoning>The premise describes a scene involving a diver and a turtle in an underwater environment. The hypothesis describes a completely different scene involving a lion and its meal at a zoo. The...


In [6]:
# Find features active in this sample
active_mask = (sample_acts > 0).any(dim=0)              # (n_features,)
active_indices = active_mask.nonzero(as_tuple=False).view(-1)

# Among active features, pick 50 with lowest corpus frequency
active_freqs = freq[active_indices]
sorted_order = torch.argsort(active_freqs)[:50]
feature_indices = active_indices[sorted_order]

selected_acts = sample_acts[:, feature_indices]          # (n_tokens, 50)

print(f"Active features in sample: {len(active_indices)}")
print(f"Selected feature indices:  {feature_indices.tolist()}")
print(f"selected_acts shape:       {selected_acts.shape}")

Active features in sample: 2141
Selected feature indices:  [34814, 13871, 15610, 54802, 16900, 35765, 2115, 19667, 146, 2035, 2032, 1235, 383, 34725, 1276, 21173, 1365, 1868, 31744, 1370, 30020, 29567, 225, 43048, 483, 4426, 49670, 637, 3770, 45552, 761, 6447, 7188, 894, 8850, 8881, 38553, 10520, 38434, 3066, 11257, 37819, 12014, 36785, 36314, 56874, 36593, 30061, 57750, 27414]
selected_acts shape:       torch.Size([145, 50])


In [7]:
np_model_id = MODEL_NAME.split("/")[-1] if "/" in MODEL_NAME else MODEL_NAME
np_sae_id = build_sae_id(sae_config)
client = NeuronpediaClient(model_id=np_model_id, sae_id=np_sae_id)

features = create_features(sample_acts, feature_indices.tolist(), tokens)
print(f"Created {len(features)} Feature objects")

for feat in features:
    feat.fetch_details(client)

print(f"Fetched Neuronpedia labels for {len(features)} features")
print(f"Example: {features[0]!r}")

Created 50 Feature objects
Fetched Neuronpedia labels for 50 features
Example: Feature(idx=34814, description='mathematical expressions and equations', max_act=627.75, n_tokens=145)


In [8]:
print("Complete input sequence (prompt + generation):")
print(sequences[sample_idx])

Complete input sequence (prompt + generation):
user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A diver is swimming with a turtle.
Hypothesis: A lion is eating raw meat at the zoo.

model 
<reasoning>The premise describes a scene involving a diver and a turtle in an underwater environment. The hypothesis describes a completely different scene involving a lion and its meal at a zoo. There is no logical connection between these two statements. Knowing that a diver is swimming with a turtle does not provide any information about whether a lion is eating meat at the zoo, nor does it contradict that possibility. Therefore, the relationship is neutral. </reasoning>
<label>neutral</label>


In [9]:
labels = {f.feature_idx: f.description for f in features}

fig = ActivationHeatmap().plot_selected_features(
    selected_acts.T,  # (n_features, n_tokens)
    tokens,
    feature_indices.tolist(),
    labels,
    title="e-SNLI Frequency Filter — Lowest-Frequency Feature Activations",
)
fig.show()

In [10]:
print(f"{'Feature IDX':<14} {'Max Activation':>16} {'Neuronpedia Label'}")
print("-" * 80)
for feat in features:
    desc = feat.description or "N/A"
    max_act = feat.max_activation()
    print(f"{feat.feature_idx:<14} {max_act:>16.4f} {desc}")

Feature IDX      Max Activation Neuronpedia Label
--------------------------------------------------------------------------------
34814                  627.7538 mathematical expressions and equations
13871                  483.8404 Chinese, Japanese, Korean, and words from other languages
15610                  514.9323 left parenthesis in math
54802                  552.8718 multilingual text fragments
16900                  486.3707 numerical identifiers
35765                  543.6511 Indic, Tamil, Amharic languages
2115                   327.2888 videos, statements, dates, #
19667                  561.5035 German words starting with Sch
146                    202.0755 names, emails, and identifiers
2035                   229.9548 multilingual specific topic phrases
2032                  1017.8450 ascii art diagrams and structures
1235                  5713.8442 random strings or repetitions
383                    253.3450 web addresses
34725                  475.6760 numbered lis

## Steering Experiment

### Load Model + SAE

In [12]:
from src.configs import ModelConfig
from src.gemma_model import GemmaModel
from src.SAE import JumpReLUSAE

model_config = ModelConfig(model_name=MODEL_NAME)
model = GemmaModel(model_config)
sae = JumpReLUSAE.from_pretrained(sae_config, device=model_config.device)

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

Load SAE resid_post/layer_40_width_65k_l0_medium/params.safetensors from google/gemma-scope-2-27b-it


### Configure Steering

In [23]:
# Specify one or more features and their steering coefficients.
# Use feature_indices from the frequency filter above, or set your own.
STEER_FEATURES = [feature_indices[0].item(), feature_indices[1].item()]  # SAE feature indices
STEER_COEFFS   = [-0.3, -0.3]   # one coefficient per feature (positive=amplify, negative=suppress)

prompt_text = sequences[sample_idx][:prompt_char_lens[sample_idx]]
inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=True)["input_ids"].to(model_config.device)

print(f"Sample index:    {sample_idx}")
print(f"Steer features:  {STEER_FEATURES}")
print(f"Steer coeffs:    {STEER_COEFFS}")
print(f"Prompt tokens:   {inputs.shape[1]}")

Sample index:    312
Steer features:  [34814, 13871]
Steer coeffs:    [-0.3, -0.3]
Prompt tokens:   99


### Baseline vs Steered Generation

In [24]:
result = model.generate_steered(
    prompt=prompt_text,
    sae=sae,
    feature_idx=STEER_FEATURES,
    coeff=STEER_COEFFS,
    target_layer=sae_config.layer,
    max_new_tokens=256,
)

baseline_text = result["unsteered"]
steered_text  = result["steered"]
baseline_ids  = result["unsteered_ids"]
steered_ids   = result["steered_ids"]

steer_label = ", ".join(f"f{fi}×{c}" for fi, c in zip(STEER_FEATURES, STEER_COEFFS))

print(f"{'PROMPT':=^80}")
print(prompt_text)
print()
print(f"{'BASELINE (unsteered)':=^80}")
print(baseline_text)
print()
print(f"{'STEERED (' + steer_label + ')':=^80}")
print(steered_text)
print()
print(f"Baseline length:  {len(baseline_ids)} tokens")
print(f"Steered length:   {len(steered_ids)} tokens")
print(f"Texts identical:  {baseline_text == steered_text}")

=====================================PROMPT=====================================
user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A diver is swimming with a turtle.
Hypothesis: A lion is eating raw meat at the zoo.

model 

==============================BASELINE (unsteered)==============================
<bos>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A diver is swimming with a turtle.
Hypothesis: A lion is eating raw meat at the zoo.

model 
<reasoning>The premise describes